In [84]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import math

num_train = 100

def feature_map(x):
    """
    Input:  [1, 28, 28]
    Output: [784, 2]
    """
    x = torch.stack(
        [
            torch.cos(math.pi * x / 2),
            torch.sin(math.pi * x / 2),
        ],
        dim=-1,
    )

    x = x.flatten(start_dim=0, end_dim=2)


    return x

# ([1,28,28],label)
full_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.Compose(
        [
            transforms.ToTensor(),
            feature_map,
        ],
    )
)
    


# select subset of MNIST
indices = torch.randperm(len(full_dataset))[:num_train]
train_dataset = Subset(full_dataset, indices)


#[64,1,28,28]
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
)

bond_dim = 10
input_dim = 2
num_sites = 784

images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)



torch.Size([64, 784, 2])
torch.Size([64])


In [85]:
import torch
import torch.nn as nn


class MPS(nn.Module):
    def __init__(self, num_sites, physical_dim=2, bond_dim=3):
        super().__init__()

        if num_sites < 2:
            raise ValueError("num_sites must be at least 2")

        self.num_sites = num_sites
        self.physical_dim = physical_dim
        self.bond_dim = bond_dim

        tensors = []

        # First site: no left bond
        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim) * 0.1
            )
        )

        # Middle sites: left and right bonds
        for _ in range(num_sites - 2):
            tensors.append(
                nn.Parameter(
                    torch.randn(physical_dim, bond_dim, bond_dim) * 0.1
                )
            )

        # Final site: no right bond
        tensors.append(
            nn.Parameter(
                torch.randn(physical_dim, bond_dim) * 0.1
            )
        )

        self.tensors = nn.ParameterList(tensors)

    def forward(self, x):
        """
        x shape: [batch_size, num_sites, physical_dim]
        """
        
        if x.ndim != 3:
            raise ValueError(
                "x must have shape [batch_size, num_sites, physical_dim]"
            )

        if x.shape[1] != self.num_sites:
            raise ValueError(
                f"Expected {self.num_sites} sites, got {x.shape[1]}"
            )

        if x.shape[2] != self.physical_dim:
            raise ValueError(
                f"Expected physical_dim={self.physical_dim}, got {x.shape[2]}"
            )

        # First site:
        # [batch, physical_dim] × [physical_dim, bond_dim]
        # → [batch, bond_dim]
        # input of first site for each batch x 
        state = x[:, 0] @ self.tensors[0]

        # Middle sites:
        # Contract the physical input and the incoming bond.
        
        # indices:
        # b - runs through Batch
        # k - runs through inputs
        # d - tensor 
        for site in range(1, self.num_sites - 1):
            state = torch.einsum(
                "bl,bp,plr->br",
                state,
                x[:, site],
                self.tensors[site]
            )

        # Final site:
        # [batch, bond_dim] contracted with the final physical input
        # and the final MPS tensor → [batch]
        y = torch.einsum(
            "bl,bp,pl->b",
            state,
            x[:, -1],
            self.tensors[-1]
        )

        return y

        

In [ ]:
model = MPS(num_sites=num_sites,physical_dim=input_dim, bond_dim=bond_dim)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(100):
    for x, labels in train_loader:
        print(x.shape)
        pred = model.forward(x)
        loss = loss_fn(pred,labels)#<--------------
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("EPOCH", epoch, loss)


torch.Size([64, 784, 2])


RuntimeError: Expected floating point type for target with class probabilities, got Long